> **公开学习副本**：保留教学代码与解释，已清除运行输出、执行计数、嵌入附件及个人元数据。完整执行原件和H02交接包保留在本地/组内私有空间。不要把本副本误认为团队已验收的新数据版本。
>
> 数据来自Olist；基础教学材料参考[CercaTrovato/olist-repurchase-analysis](https://github.com/CercaTrovato/olist-repurchase-analysis)，保留上游代码与数据许可。仅在本地有原始数据时运行；输出可能包含逐行客户数据，保持在被Git忽略的outputs目录。源码中的定义及限制仍应阅读。
> 岗位学习课：输出隔离到outputs/public_learning/，版本带public-learning，动态选取示例客户。不能替换已发给团队的H02_candidate_v2。按02→03→04运行。

# H02 候选交接 从历史订单到特征与开放标签

前一课 02 已完成订单事实表。本课实际执行岗位第 2、3 项，并导出 H02 候选文件。采用用户授权的已探索三期：2017-10-01、2018-01-01、2018-04-01。第三期明确为已见探索期，不能作为未触碰的最终测试。

每节顺序是问题、代码、输出与解释。特征只用快照前六个自然月，标签使用快照起未来 90 天。本次三个日期的结果都已开放过，故标签可以用于候选包的探索复算。最终评价日期未核定，无新的最终评价标签生成或开放。

本包是待团队验收的候选版本。观察边界暂用 [2017-01-01,2018-09-01)，这是现有覆盖诊断支持的工作范围，仍缺源系统采集完整性的独立证明。下游使用时保留这个限制。

## 1 输入与规则版本

先核对第 1 项的事实表哈希和原始文件哈希，避免接错版本。日期设计由配置明确列出。换成未核定日期会在后面阻止开放标签。

In [ ]:
from pathlib import Path
import hashlib, json, sys, shutil
from datetime import datetime, timezone
from zipfile import ZipFile, ZIP_DEFLATED
import pandas as pd
import numpy as np
from IPython.display import display

ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'data/source-manifest.json').is_file())
STEP1=ROOT/'outputs/public_learning/role01_step1_v1'
OUT=ROOT/'outputs/public_learning/H02_candidate_v1'
OUT.mkdir(parents=True,exist_ok=True)
AUDIT=OUT/'audit'; AUDIT.mkdir(exist_ok=True)
VERSION='H02-public-learning-v1'
KEY=['customer_unique_id','snapshot_date']
FEATURES=['R','F','M','average_order_value','max_order_value','average_item_rows','freight_share','observed_history_days','recent30_orders','recent30_spend','state']
SPEC=[('2017-10-01','development_train'),('2018-01-01','development_validation'),('2018-04-01','explored_2018_04')]
OPEN_DATES={'2017-10-01','2018-01-01','2018-04-01'}
OBS_START=pd.Timestamp('2017-01-01'); OBS_END=pd.Timestamp('2018-09-01')

def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''): h.update(block)
    return h.hexdigest()

previous=json.loads((STEP1/'step1-manifest.json').read_text(encoding='utf-8'))
for item in previous['artifacts']:
    assert sha(STEP1/item['file'])==item['sha256'], '第 1 项产物被修改，请重新运行 02'
source=json.loads((ROOT/'data/source-manifest.json').read_text(encoding='utf-8'))
assert sha(ROOT/'data/olist-v2.zip')==source['zip_sha256']
for item in previous['source_files']:
    assert sha(ROOT/'data/raw'/item['file'])==item['sha256']
orders=pd.read_csv(STEP1/'order_fact_audit.csv',dtype={'order_id':'string','customer_unique_id':'string','customer_id':'string'})
for col in ['order_purchase_timestamp','order_approved_at','order_delivered_customer_date']:
    before=orders[col].isna()
    orders[col]=pd.to_datetime(orders[col],errors='coerce')
    assert not (orders[col].isna() & ~before).any()
eligible=orders.loc[orders.order_approved_at.notna() & orders.amount.gt(0)].copy()
assert eligible.order_id.is_unique
pd.set_option('display.max_columns',30)
display(pd.DataFrame(SPEC,columns=['snapshot_date','split']))
print('本次公开范围仅限已见探索日期。最终评价方案待 3、6 号核定。')

## 2 检查所有日期的覆盖

分别统计购买、批准和合格批准订单，补齐中间没有记录的日期。零记录日的原因需要核查，不能仅凭它证明源系统丢数。月度统计保留 2016 年起的完整范围，让尾部和早期稀疏可见。

每日覆盖文件用于复核连续性；每期覆盖表会记录历史和标签区间的零记录日数、最长连续零记录日数和窗口边界。

In [ ]:
calendar=pd.date_range(orders.order_purchase_timestamp.min().normalize(),orders.order_purchase_timestamp.max().normalize(),freq='D')
daily=pd.DataFrame(index=calendar)
for name,values in [('purchase_orders',orders.order_purchase_timestamp),('approved_orders',orders.order_approved_at),('eligible_approved_orders',eligible.order_approved_at)]:
    daily[name]=values.dt.normalize().value_counts().reindex(calendar,fill_value=0).astype(int)
daily.index.name='date'
monthly=daily.groupby(daily.index.to_period('M')).sum()
monthly.index=monthly.index.astype(str); monthly.index.name='month'
display(monthly)
print('批准时间最大值', orders.order_approved_at.max())
print('暂定观察结束日不含', OBS_END.date())
print('尾部逐日检查')
display(daily.loc['2018-08-20':])
gap_orders=orders.loc[orders.order_purchase_timestamp.ge('2018-04-21') & orders.order_purchase_timestamp.lt('2018-04-24')].copy()
gap_approval_counts=(gap_orders.order_approved_at.dt.strftime('%Y-%m-%d').fillna('MISSING')
                     .value_counts().sort_index().rename_axis('approval_date').reset_index(name='orders'))
print('4月21至23日下单订单随后批准日期；先调查零记录日，不直接补零或删除')
display(gap_approval_counts)

def coverage_stats(start,end):
    counts=daily['eligible_approved_orders'].reindex(pd.date_range(start.normalize(),end.normalize()-pd.Timedelta(days=1)),fill_value=0)
    zero=counts.eq(0)
    runs=zero.groupby(zero.ne(zero.shift()).cumsum()).sum()
    return {'days':len(counts),'zero_days':int(zero.sum()),'max_zero_run':int(runs.max()) if len(runs) else 0}

2018 年 9 月批准记录骤减，10 月无批准记录。我们没有把晚到的一笔记录当作整个 9 月完整覆盖的证据。当前开放三期的标签最晚到 2018-06-30 之前，距离尾部还有余量；这仍是有条件的观察范围判断。

另一个发现：2018-04-21 至 23 日批准数为 0，但这三天有 641 笔购买记录。追查发现，其中 614 笔在 24 日批准，26 笔在 25 日批准，1 笔在 27 日批准，批准时间都不缺失。这说明零批准日与零购买日不同；记录呈现延后集中批准现象，不能据此断定全库丢单或确定业务原因。暂按真实记录的批准时间归窗，保留这个异常，不用购买日替换批准日。4 月快照保留覆盖待核提示，只作探索。

## 3 先列时间设计 再计算标签

历史区间 [t-6 个自然月,t)，未来区间 [t,t+90 天)。六个月不是固定 180 天，90 天也不是三个自然月。训练标签结束时间不得晚于后续评分时点。本次没有设置额外入库延迟，使用批准事件及时记录的假设。

不满足工作观察范围的日期只列为待核，禁止用全零标签替代。本格也列一个超出范围的示例日期，但不会为它生成标签。

In [ ]:
window_rows=[]
for date,split in SPEC:
    cut=pd.Timestamp(date); start=cut-pd.DateOffset(months=6); end=cut+pd.Timedelta(days=90)
    assert date in OPEN_DATES, '未确认开放的日期不能自动输出标签'
    assert start>=OBS_START and end<=OBS_END, '窗口超出暂定覆盖范围'
    row={'snapshot_date':date,'split':split,'history_start':str(start.date()),'history_end_exclusive':date,'label_start':date,'label_end_exclusive':str(end.date()),
         'label_mature_at':str(end.date()),'coverage_status':'within_working_bounds_source_completeness_unproven',
         'coverage_warning':'2018-04-21_to_23_zero_approvals_with_purchases_followup_pending' if date=='2018-04-01' else 'no_zero_event_days_in_selected_intervals_not_completeness_proof',
         'prior_exploration':True,'formal_final_test':False}
    row.update({'history_'+k:v for k,v in coverage_stats(start,cut).items()})
    row.update({'label_'+k:v for k,v in coverage_stats(cut,end).items()})
    window_rows.append(row)
windows=pd.DataFrame(window_rows)
for first,second in zip(window_rows,window_rows[1:]):
    assert pd.Timestamp(first['label_mature_at'])<=pd.Timestamp(second['snapshot_date'])
display(windows)
invalid_cut=pd.Timestamp('2018-07-01')
display(pd.DataFrame([{'snapshot_date':str(invalid_cut.date()),'label_end_exclusive':str((invalid_cut+pd.Timedelta(days=90)).date()),
                       'decision':'超出工作边界，不生成标签'}]))

## 4 定义历史特征和标签的计算函数

先固定历史名单，再求未来是否有同口径事件。未来新买家不会被加进名单。最近 30 天没有事件可以记活动数 0；无明细金额在第 1 项保持未知并按资格排除。

R 按秒差除 86400 保留小数。F 为不同订单数，M 只含商品金额。运费比例为 sum(freight)/(M+sum(freight))。observed_history_days 只描述窗口内最早记录，不是客户注册年龄。state 按历史批准时间及订单号排序取最后非空州。

In [ ]:
def make_snapshot(events, cut):
    cut=pd.Timestamp(cut); start=cut-pd.DateOffset(months=6); end=cut+pd.Timedelta(days=90)
    if start<OBS_START or end>OBS_END:
        raise ValueError('未满足暂定历史与后续覆盖范围，不能生成零标签')
    hist=events.loc[events.order_approved_at.ge(start)&events.order_approved_at.lt(cut)].sort_values(['order_approved_at','order_id'])
    future=events.loc[events.order_approved_at.ge(cut)&events.order_approved_at.lt(end)]
    grouped=hist.groupby('customer_unique_id')
    x=grouped.agg(first=('order_approved_at','min'),last=('order_approved_at','max'),
                  F=('order_id','nunique'),M=('amount','sum'),max_order_value=('amount','max'),
                  average_item_rows=('item_rows','mean'),freight_sum=('freight','sum'))
    x['R']=(cut-x['last']).dt.total_seconds()/86400
    x['observed_history_days']=(cut-x['first']).dt.total_seconds()/86400
    x['average_order_value']=x.M/x.F
    x['freight_share']=x.freight_sum/(x.M+x.freight_sum)
    recent=hist.loc[hist.order_approved_at.ge(cut-pd.Timedelta(days=30))].groupby('customer_unique_id').agg(
        recent30_orders=('order_id','nunique'),recent30_spend=('amount','sum'))
    x=x.join(recent).fillna({'recent30_orders':0,'recent30_spend':0})
    x['recent30_orders']=x.recent30_orders.astype(int)
    x['state']=grouped.customer_state.last().fillna('UNKNOWN')
    y=pd.Series(x.index.isin(future.customer_unique_id).astype(int),index=x.index,name='y')
    assert np.isfinite(x[FEATURES[:-1]].to_numpy()).all()
    assert x.F.sum()==len(hist)
    assert np.isclose(x.M.sum(),hist.amount.sum(),rtol=1e-12,atol=1e-7)
    return x[FEATURES].reset_index(), y.reset_index(), hist, future

feature_parts=[]; label_parts=[]; cohort_rows=[]
for row in window_rows:
    date=row['snapshot_date']; split=row['split']; cut=pd.Timestamp(date); start=pd.Timestamp(row['history_start'])
    x,y,h,f=make_snapshot(eligible,cut)
    for frame in [x,y]:
        frame['snapshot_date']=date; frame['split']=split; frame['data_version']=VERSION
    feature_parts.append(x); label_parts.append(y)
    before=orders.loc[orders.order_approved_at.ge(start)&orders.order_approved_at.lt(cut)]
    target_buyers=set(f.customer_unique_id)
    positives_before=len(set(before.customer_unique_id)&target_buyers)
    cohort_rows.append({**row,'history_orders_before_amount_rule':len(before),'customers_before_amount_rule':before.customer_unique_id.nunique(),
                       'history_orders':len(h),'customers':len(x),'positives_before_amount_rule':positives_before,
                       'repurchase_rate_before_amount_rule':positives_before/before.customer_unique_id.nunique(),
                       'positives':int(y.y.sum()),'negatives':int(len(y)-y.y.sum()),'repurchase_rate':float(y.y.mean()),
                       'customers_excluded_by_amount_rule':int(before.customer_unique_id.nunique()-len(x))})
features=pd.concat(feature_parts,ignore_index=True)[KEY+['split','data_version']+FEATURES]
labels=pd.concat(label_parts,ignore_index=True)[KEY+['split','data_version','y']]
cohorts=pd.DataFrame(cohort_rows)
display(cohorts[['snapshot_date','split','history_orders','customers','positives','repurchase_rate','history_zero_days','label_zero_days']])
display(features.head())

## 5 核对键 连接范围与跨期重复客户

特征与开放标签分别检查组合键。外连接 indicator 同时检查只在左、只在右、两侧都有的键，确认不会因为内连接静默丢人。

同一个买家可以在多个快照出现，所以跨期连接必须同时带 snapshot_date。这种重复评分场景需要披露相关性，不能视为三次独立试验。

In [ ]:
assert not features.duplicated(KEY).any() and not labels.duplicated(KEY).any()
joined_keys=features[KEY+['split','data_version']].merge(labels,on=KEY+['split','data_version'],how='outer',validate='one_to_one',indicator=True)
assert joined_keys['_merge'].eq('both').all()
assert 'y' not in features.columns
overlap=[]
for i,(a,_) in enumerate(SPEC):
    for b,_ in SPEC[i+1:]:
        left=set(features.loc[features.snapshot_date.eq(a),'customer_unique_id'])
        right=set(features.loc[features.snapshot_date.eq(b),'customer_unique_id'])
        overlap.append({'left_snapshot':a,'right_snapshot':b,'shared_buyers':len(left&right)})
display(pd.DataFrame(overlap))
print('特征和标签双向匹配', len(joined_keys), '行；不同买家',features.customer_unique_id.nunique())

## 6 检查预测时是否已经知道送达与评价

在每期历史合格订单中，统计快照时仍未发生的实际送达，以及在快照或之后才答复的评价。评价用原始 review_answer_timestamp 检查；多评在这里只用于审计，不接进特征。

本版十一项特征均从历史批准订单金额、条数、时间及历史客户州构成。没有使用最终状态、实际交付结果或评分作为输入。静态 CSV 缺少字段变更日志，金额和州在批准后可用且未发生影响研究的追溯修改，是仍需披露的假设。

In [ ]:
review_name='olist_order_reviews_dataset.csv'
review_expected=next(r['sha256'] for r in source['files'] if r['name']==review_name)
assert sha(ROOT/'data/raw'/review_name)==review_expected
reviews=pd.read_csv(ROOT/'data/raw'/review_name,dtype={'order_id':'string'})
raw_answers=reviews.review_answer_timestamp.copy()
reviews['review_answer_timestamp']=pd.to_datetime(raw_answers,errors='coerce')
assert not (raw_answers.notna()&reviews.review_answer_timestamp.isna()).any()
availability=[]
for date,split in SPEC:
    cut=pd.Timestamp(date); _,_,h,_=make_snapshot(eligible,cut)
    relevant_reviews=reviews.loc[reviews.order_id.isin(h.order_id)]
    availability.append({'snapshot_date':date,'history_orders':len(h),
        'delivered_at_or_after_snapshot':int(h.order_delivered_customer_date.ge(cut).sum()),
        'delivery_date_missing':int(h.order_delivered_customer_date.isna().sum()),
        'review_records_answered_at_or_after_snapshot':int(relevant_reviews.review_answer_timestamp.ge(cut).sum()),
        'use_delivery_or_review_as_features':False})
display(pd.DataFrame(availability))

## 7 人工边界用例与未来变化检查

在人工记录中放入历史起点之前一秒、历史起点、快照之前一秒、快照时点、标签结束之前一秒和标签结束时点。边界预期可以在计算前手工写出。它们是测试样本，不混入真实输出。

再改变未来订单的金额与买家归属，确认历史特征保持不变。最后测试覆盖不足会报错而非产生零标签。

In [ ]:
cut=pd.Timestamp('2018-01-01'); start=cut-pd.DateOffset(months=6); end=cut+pd.Timedelta(days=90)
times=[start-pd.Timedelta(seconds=1),start,cut-pd.Timedelta(seconds=1),cut,end-pd.Timedelta(seconds=1),end]
expected_history=[False,True,True,False,False,False]
expected_future=[False,False,False,True,True,False]
fixture=pd.DataFrame({'order_id':[f'test-{i}' for i in range(6)],'customer_unique_id':['test-buyer']*6,
                      'order_approved_at':times,'amount':[1.,10.,20.,30.,40.,50.],
                      'freight':[1.]*6,'item_rows':[1]*6,'customer_state':['SP']*6})
fx,fy,fh,ff=make_snapshot(fixture,cut)
boundary=pd.DataFrame({'event_time':times,'expected_history':expected_history,'expected_future':expected_future})
boundary['actual_history']=fixture.order_id.isin(fh.order_id)
boundary['actual_future']=fixture.order_id.isin(ff.order_id)
assert boundary.actual_history.tolist()==expected_history
assert boundary.actual_future.tolist()==expected_future
assert fx.iloc[0].F==2 and fx.iloc[0].M==30 and fy.iloc[0].y==1
changed=fixture.copy(); future_mask=changed.order_approved_at.ge(cut)
changed.loc[future_mask,'amount']=99999
changed.loc[future_mask,'customer_unique_id']='future-new-buyer'
new_x,new_y,_,_=make_snapshot(changed,cut)
pd.testing.assert_frame_equal(fx,new_x)
assert new_y.iloc[0].y==0
try:
    make_snapshot(fixture,'2018-07-01')
    raise AssertionError('覆盖不足未被阻止')
except ValueError:
    pass
display(boundary)
print('边界、未来变化和覆盖不足拒绝检查通过。')

## 8 用一笔真实订单检验快照时点

选已知有历史订单的买家，取运行时所选一笔订单的实际批准时刻作为临时测试分界。将分界从该时刻移后一秒，订单会从未来侧移到历史侧，客户 F 应增加 1。这只用于验证代码，正式快照仍然是配置中的三个日期。

In [ ]:
past_ids=set(eligible.loc[eligible.order_approved_at.ge('2017-08-01') & eligible.order_approved_at.lt('2018-01-01'),'customer_unique_id'])
jan_ids=set(eligible.loc[eligible.order_approved_at.ge('2018-01-01') & eligible.order_approved_at.lt('2018-02-01'),'customer_unique_id'])
buyer_id=sorted(past_ids & jan_ids)[0]
buyer_events=eligible.loc[eligible.customer_unique_id.eq(buyer_id)].sort_values('order_approved_at')
boundary_order=buyer_events.loc[buyer_events.order_approved_at.ge('2018-01-01')].iloc[0]
real_cut=boundary_order.order_approved_at
real_x,_,real_h,real_f=make_snapshot(eligible,real_cut)
shift_x,_,shift_h,_=make_snapshot(eligible,real_cut+pd.Timedelta(seconds=1))
assert boundary_order.order_id not in set(real_h.order_id)
assert boundary_order.order_id in set(real_f.order_id)
assert boundary_order.order_id in set(shift_h.order_id)
old_f=real_x.set_index('customer_unique_id').loc[buyer_id,'F']
new_f=shift_x.set_index('customer_unique_id').loc[buyer_id,'F']
assert new_f==old_f+1
real_boundary=pd.DataFrame([{'order_id':boundary_order.order_id,'customer_unique_id':buyer_id,
                             'approval_time':real_cut,'at_cut_in_history':False,'at_cut_in_future':True,
                             'one_second_later_in_history':True,'F_at_cut':old_f,'F_one_second_later':new_f}])
display(real_boundary)

## 9 随机客户及多单客户从原始明细复算

每期固定随机抽三名客户，再补一个多单客户和一个正例。所有案例都是已开放的探索日期。

直接从原始 items 取对应明细重算 M 和运费，从原始订单批准时间重算 R、F 与未来标签。输出订单证据区分 history_input 和 future_label_evidence，方便你自己核对。

In [ ]:
raw_items=pd.read_csv(ROOT/'data/raw/olist_order_items_dataset.csv',dtype={'order_id':'string'})
trace_checks=[]; trace_events=[]
for date,split in SPEC:
    x,y,h,f=make_snapshot(eligible,date); lookup=x.merge(y,on='customer_unique_id',validate='one_to_one')
    selected=list(lookup.sample(n=3,random_state=6400).customer_unique_id)
    selected.append(lookup.loc[lookup.F.gt(1),'customer_unique_id'].iloc[0])
    selected.append(lookup.loc[lookup.y.eq(1),'customer_unique_id'].iloc[0])
    for buyer in dict.fromkeys(selected):
        hist=h.loc[h.customer_unique_id.eq(buyer)]; future=f.loc[f.customer_unique_id.eq(buyer)]
        detail=raw_items.loc[raw_items.order_id.isin(hist.order_id)]
        row=lookup.loc[lookup.customer_unique_id.eq(buyer)].iloc[0]
        assert hist.order_id.nunique()==row.F
        assert np.isclose(detail.price.sum(),row.M,rtol=1e-12,atol=1e-7)
        assert np.isclose(detail.freight_value.sum()/(detail.price.sum()+detail.freight_value.sum()),row.freight_share)
        assert np.isclose((pd.Timestamp(date)-hist.order_approved_at.max()).total_seconds()/86400,row.R)
        assert int(len(future)>0)==row.y
        trace_checks.append({'snapshot_date':date,'customer_unique_id':buyer,'F':int(row.F),'M':row.M,'R':row.R,'y':int(row.y),'check':'passed'})
        for evidence,role in [(hist,'history_input'),(future,'future_label_evidence')]:
            part=evidence[['order_id','customer_unique_id','order_approved_at','amount','freight','item_rows']].copy()
            part['snapshot_date']=date;part['event_role']=role;trace_events.append(part)
trace_checks=pd.DataFrame(trace_checks);trace_events=pd.concat(trace_events,ignore_index=True)
display(trace_checks)

## 10 导出特征 标签 覆盖和核查证据

特征和 y 分成两个文件。明确 features 列清单供 3、4 号选择输入，客户键、split、版本均为元数据。全部开放标签只对应本次三期已见探索日期。

先保存 CSV，再回读检查值和键。产物清单最后生成，交接者应核对哈希后使用，复跑中断的部分文件不得视为验收通过。

In [ ]:
def save(df,path):
    temp=path.with_suffix(path.suffix+'.partial')
    df.to_csv(temp,index=False,encoding='utf-8-sig',date_format='%Y-%m-%d %H:%M:%S')
    temp.replace(path)

save(features,OUT/'snapshot_features.csv');save(labels,OUT/'snapshot_labels.csv')
save(cohorts,OUT/'coverage-and-cohorts.csv')
save(daily.reset_index(),AUDIT/'daily-coverage.csv');save(monthly.reset_index(),AUDIT/'monthly-coverage.csv')
save(gap_approval_counts,AUDIT/'april-gap-purchase-cohort-approval-dates.csv')
save(pd.DataFrame(overlap),AUDIT/'buyer-overlap.csv');save(pd.DataFrame(availability),AUDIT/'field-time-audit.csv')
save(boundary,AUDIT/'synthetic-boundaries.csv');save(real_boundary,AUDIT/'real-order-boundary.csv')
save(trace_checks,AUDIT/'customer-trace-checks.csv');save(trace_events,AUDIT/'customer-trace-orders.csv')
for name in ['stage-audit.csv','relation-checks.csv','amount-reconciliation.csv','customer-filter-counts.csv','excluded-orders.csv']:
    shutil.copy2(STEP1/name,AUDIT/('step1-'+name))
loaded_x=pd.read_csv(OUT/'snapshot_features.csv',dtype={'customer_unique_id':'string'})
loaded_y=pd.read_csv(OUT/'snapshot_labels.csv',dtype={'customer_unique_id':'string'})
assert loaded_x[KEY].astype(str).equals(features[KEY].astype(str))
assert loaded_y[KEY].astype(str).equals(labels[KEY].astype(str))
np.testing.assert_allclose(loaded_x[FEATURES[:-1]],features[FEATURES[:-1]],rtol=1e-12,atol=1e-9)
assert loaded_x.state.tolist()==features.state.tolist()
assert loaded_y.y.tolist()==labels.y.tolist()
assert set(loaded_y.snapshot_date)==OPEN_DATES
(OUT/'feature-columns.json').write_text(json.dumps({'features':FEATURES,'target':'y','join_key':KEY,'metadata':['split','data_version']},indent=2),encoding='utf-8')
print('导出与回读通过，共',len(features),'条客户快照。')

## 11 字段字典 交接单与最终标签管理

正式最终评价方案尚未核定。本次最终标签状态为未生成，而不是“已经封存好某个新测试集”。未来选定日期后，先记录是否用于过选模、观察范围、标签结束与开放条件，再由 1 号单独保管。

下面会把缺口、下一次交付触发条件和影响下游范围写进 HANDOFF.md。Slides、英文讲稿和报告要点在分析结果核定后完成，此候选数据包并不代表这些材料已经完成。

In [ ]:
definitions=[
('R','float / 天','(快照-历史最近批准时间)秒数/86400'),('F','int / 笔','历史不同 order_id 数'),
('M','float / BRL','历史订单商品 amount 之和，不含运费'),('average_order_value','float / BRL','M/F'),
('max_order_value','float / BRL','历史单笔 amount 最大值'),('average_item_rows','float / 条每单','历史 item_rows 平均值'),
('freight_share','float / 比例','历史运费/(M+历史运费)'),('observed_history_days','float / 天','快照距窗口内最早批准时间，不是注册年龄'),
('recent30_orders','int / 笔','[t-30天,t) 内历史不同订单数，无事件为0'),('recent30_spend','float / BRL','同近期区间商品额，无事件为0'),
('state','string / 州','历史按批准时间和order_id排序，取最后非空客户州；未知回退UNKNOWN')]
dictionary='# H02 候选字段字典\n\n版本 '+VERSION+'。一行一客户快照。主键 customer_unique_id + snapshot_date。\n\n'
dictionary+='customer_unique_id 为匿名买家字符串；snapshot_date 为 YYYY-MM-DD；split 为开发训练、开发验证或已见探索标记；data_version 为数据版本。四者均为元数据。\n\n'
dictionary+='| 字段 | 类型与单位 | 公式 |\n|---|---|---|\n'
dictionary+='\n'.join('|'+ '|'.join(row)+'|' for row in definitions)
dictionary+='\n\n所有特征来自 [t-6个自然月,t) 内批准且正商品金额订单。金额、运费和州采用批准后可用及静态导出值适用的假设，无法从修改日志验证。\n\ny 为 int 0/1，表示 [t,t+90天) 是否出现同口径事件。0 只在工作观察范围通过时生成；源系统完整性尚无独立证明。标签仅含已开放探索日期。\n\ncoverage-and-cohorts.csv 记录起止、成熟时点、历史及标签零记录日与最长零记录段、金额规则前后订单/客户/正例/比例。daily-coverage 与 monthly-coverage 区分购买、批准、合格批准事件。计数中零记录不直接等于源数据缺失。\n'
dictionary+='\n字段来源：订单及时间来自 orders.order_id / order_approved_at；匿名买家和州来自 customers.customer_unique_id / customer_state，按 customer_id 连接；amount 为 items.price 按 order_id 求和，freight 为 items.freight_value 求和，item_rows 为订单明细行数。R、observed_history_days、recent30_orders 使用批准时间；F 使用订单号；所有金额特征使用商品金额或运费；state 使用历史客户州。来源均先由02汇总到订单，再由03聚合到客户快照。\n\n注意：positive_before_amount_rule 固定未来合格事件定义，仅放宽历史名单的商品金额资格；不是把未来标签也改成无金额条件的另一套任务。各列的实际名字以CSV为准（positives_before_amount_rule）。\n'
(OUT/'data-dictionary.md').write_text(dictionary,encoding='utf-8')
handoff=f"""# H02 候选交接单

[H02] 1号 → 2至6号 | {VERSION} | 待本人核验与团队验收

用户已授权先沿用已探索三期制作候选版本。当前不是已批准的最终时间方案。

输入为仓库原始 ZIP、来源清单和通过哈希核对的 step1 订单事实表。生成版本及文件哈希见 data-manifest.json。

交付 snapshot_features.csv、snapshot_labels.csv、data-dictionary.md、coverage-and-cohorts.csv、feature-columns.json、audit、reproduce。

共 {len(features):,} 条快照，{features.customer_unique_id.nunique():,} 个买家。分期人数、筛选前后正例和比例见 coverage-and-cohorts.csv。特征与标签双向全量核键通过。

已核：原始键与连接、订单金额、工作范围内历史与标签区间、每日每月记录、历史字段时间审计、人工边界、真实边界订单、随机与多单客户追溯、CSV回读。

待核：H01正式日期和最终评价设计；源系统采集完整性；真实运营字段入库延迟。2018-04-01已经看过探索结果。三期均不得重新宣称未触碰测试。

覆盖异常：2018-04-21至23日没有批准记录，但有641笔下单。它们随后分别于24日批准614笔、25日26笔、27日1笔。暂按批准时间保留，原因与源系统完整性仍待核；详见audit/april-gap-purchase-cohort-approval-dates.csv。4月快照只供探索复算，不标为正式已验证完整窗口。

最终评价标签尚未生成。后续由1号在单独私有目录保管，仅在方法冻结且团队明确开放后交付。新增日期默认不在当前 OPEN_DATES 中，生成开放标签会被阻止。

请3、6号核日期、资格和最终评价安排；2号核缺失与分布；3、4号核同一客户快照键与版本；5号按该键对齐预测和群标记。本人核验后在组内私有空间交接，接收方回复已验收、退回补件或暂收部分可用。

下一次交付：收到H01正式日期及标签开放安排后更新版本并重建；具体时刻待1、3、6号约定。若窗口或资格变化，H03 EDA、H04预测、H05分群、H06反馈、H07数字材料均需核受影响范围并重跑。

11月13日 Slides、英文讲稿和报告要点待正式分析结果核定后完成。
"""
(OUT/'HANDOFF.md').write_text(handoff,encoding='utf-8')
(OUT/'FINAL_LABEL_POLICY.md').write_text('# 最终评价标签状态\n\n当前状态：最终日期未核定，标签未生成。公开包只含已经查看过的三期探索标签。\n\n新最终标签应保存在候选包之外的1号私有目录。冻结前，逐行标签与正例数量不写入共享文件、Notebook输出、图表、日志或压缩包。1号独立核查后记录生成哈希和开放条件。换日期需要更新配置与代码，不能把旧探索标签重新命名为最终测试。\n',encoding='utf-8')

## 12 封装复跑入口与文件哈希

候选包包含两份完整教学 Notebook 和生成入口。复跑从项目根目录执行，先重建第 1 项，再运行本课。每个文件用 SHA-256 指纹校验；清单自身不写入自身哈希，压缩包会包括完整清单。

阅读次序建议为第 3 节时间表、第 4 节计算函数、第 7 至 9 节核查，最后看实际输出。你应能自己复算一个客户，再确认交接单中的待核项。

In [ ]:
reproduce=OUT/'reproduce';(reproduce/'notebooks').mkdir(parents=True,exist_ok=True)
for name in ['02-role01-order-fact-and-history.ipynb','03-role01-h02-coverage-features-labels.ipynb']:
    raw=json.loads((ROOT/'notebooks'/name).read_text(encoding='utf-8'))
    for cell in raw['cells']:
        if cell['cell_type']=='code':cell['outputs']=[];cell['execution_count']=None
    (reproduce/'notebooks'/name).write_text(json.dumps(raw,ensure_ascii=False,indent=1),encoding='utf-8')
shutil.copy2(ROOT/'scripts/build_h02_candidate.py',reproduce/'build_h02_candidate.py')
shutil.copy2(ROOT/'requirements.txt',reproduce/'requirements.txt')
shutil.copy2(ROOT/'data/source-manifest.json',reproduce/'source-manifest.json')
command='.\\.venv312\\Scripts\\python.exe scripts/build_h02_candidate.py'
(OUT/'RUN.md').write_text('# 复跑候选 H02\n\n在完整仓库根目录运行：\n\n```powershell\n'+command+'\n```\n\n需已安装仓库 requirements.txt，注册 olist312 内核并完成原始 ZIP 校验解压。入口依次执行02与03并保存输出；失败日志位于 outputs/public_learning/H02_build_failure.log。\n\n在另一份同源仓库中，把 reproduce/notebooks 中两份文件放到 notebooks，把 reproduce/build_h02_candidate.py 放到 scripts，再运行同一命令。原始输入来自该仓库 data/raw，所有文件先核哈希。\n\n本配置仅开放已探索三期。新增最终评价日期须另定保管与开放流程。每次复跑后先核 manifest 哈希，再交接。\n',encoding='utf-8')
artifacts=[]
for path in sorted(OUT.rglob('*')):
    if path.is_file() and path.name!='data-manifest.json' and not path.name.endswith('.partial'):
        artifacts.append({'file':path.relative_to(OUT).as_posix(),'bytes':path.stat().st_size,'sha256':sha(path)})
manifest={'data_version':VERSION,'status':'candidate_pending_H01_and_team_receipt','generated_at_utc':datetime.now(timezone.utc).isoformat(),
          'source_zip_sha256':source['zip_sha256'],'step1_manifest_sha256':sha(STEP1/'step1-manifest.json'),
          'input_hashes':previous['source_files']+[{'file':review_name,'sha256':review_expected}],
          'reference_protocol_sha256':sha(ROOT/'docs/role01/protocol-reference.md'),
          'runtime':{'python':sys.version.split()[0],'pandas':pd.__version__,'numpy':np.__version__},
          'observation_start':str(OBS_START.date()),'observation_end_exclusive':str(OBS_END.date()),
          'coverage_assumption':'working_bounds_not_source_completeness_proof','history_months':6,'target_days':90,
          'snapshot_summary':cohorts.to_dict('records'),'open_dates':sorted(OPEN_DATES),
          'final_evaluation':{'date':None,'labels_generated':False,'release_status':'not_selected_not_released'},
          'tests':{'key_alignment':'passed','csv_roundtrip':'passed','boundary_fixture':'passed','real_boundary_order':'passed',
                   'future_mutation':'passed','insufficient_followup_rejected':'passed','customer_trace_cases':len(trace_checks)},
          'feature_columns':FEATURES,'join_key':KEY,'raw_orders':len(orders),'eligible_orders':len(eligible),
          'excluded_orders':len(orders)-len(eligible),'customer_snapshot_rows':len(features),
          'unique_buyers':int(features.customer_unique_id.nunique()),'generation_command':command,
          'artifact_hash_note':'all listed artifacts; manifest excludes its own hash','artifacts':artifacts}
(OUT/'data-manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
for item in artifacts: assert sha(OUT/item['file'])==item['sha256']
zip_path=ROOT/'outputs/public_learning/H02_candidate_v1.zip'
with ZipFile(zip_path,'w',ZIP_DEFLATED) as archive:
    for item in artifacts:archive.write(OUT/item['file'],arcname='H02_candidate_v1/'+item['file'])
    archive.write(OUT/'data-manifest.json',arcname='H02_candidate_v1/data-manifest.json')
with ZipFile(zip_path) as archive:assert archive.testzip() is None
display(cohorts[['snapshot_date','split','customers','positives','repurchase_rate']])
print('候选交接包',zip_path)
print('正式日期、最终评价标签与团队验收仍待核定。')